# Lost Wells — Colab Master Guide

This is the educational setup, validation, and final-merge notebook. Full inference runs in the four state worker notebooks.

**Meaning of an output:** the U-Net finds printed well symbols; a result is saved only when its nearest loaded documented-well coordinate is more than 100 m away. It is a candidate undocumented location, not a confirmed well.

## Architecture from first principles

1. A scanned map is a pixel grid.
2. The pretrained U-Net assigns each pixel a well-symbol probability.
3. Adjacent positive pixels form a connected component; tiny components are rejected.
4. GeoTIFF metadata converts the component center from pixels to WGS84 coordinates.
5. The nearest documented registry point is measured with a great-circle distance.
6. Distances over 100 m are retained as candidates.
7. A separate 60 m merge step removes repeated model detections without deleting the original evidence.

## 1. Optional master-notebook runtime setup

The state workers contain the same installer. For validation or pilot work here, choose **Runtime → Change runtime type → T4 GPU**, run this cell, then choose **Runtime → Restart session**.

In [ ]:
import os, subprocess, sys, importlib.metadata as metadata
os.environ["TF_USE_LEGACY_KERAS"] = "1"
os.environ["SM_FRAMEWORK"] = "tf.keras"

def run(command):
    print("RUN:", " ".join(command))
    result = subprocess.run(command, text=True, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT)
    print(result.stdout[-12000:])
    if result.returncode:
        raise RuntimeError(f"command failed with exit code {result.returncode}")

run(["apt-get", "-qq", "update"])
run(["apt-get", "-qq", "install", "-y", "gdal-bin", "libgdal-dev"])
tf_version = metadata.version("tensorflow")
print("Installed TensorFlow:", tf_version)
if not tf_version.startswith("2.20."):
    raise RuntimeError(f"Expected Colab TensorFlow 2.20.x, found {tf_version}; stop and report this")
run([sys.executable, "-m", "pip", "install", "--upgrade",
     "numpy==1.26.4", "opencv-python-headless==4.10.0.84"])
run([sys.executable, "-m", "pip", "install", "--upgrade", "tf_keras==2.20.1"])
run([sys.executable, "-m", "pip", "install", "--upgrade", "--no-deps",
     "keras_applications==1.0.8", "image-classifiers==1.0.0",
     "efficientnet==1.1.1", "segmentation-models==1.0.1"])
run([sys.executable, "-m", "pip", "install", "--upgrade",
     "simplekml==1.3.6", "geopandas", "requests"])
gdal_version = subprocess.check_output(["gdal-config", "--version"], text=True).strip()
run([sys.executable, "-m", "pip", "install", f"GDAL=={gdal_version}"])
print("Install complete. Use Runtime > Restart session before continuing.")

## 2. Mount Drive, update the repository, and verify inputs

Run after restarting. This master notebook never calls the rate-limited EDX API. It uses the files already stored in Drive.

In [ ]:
import os, subprocess, sys, csv
from pathlib import Path
os.environ["TF_USE_LEGACY_KERAS"] = "1"
os.environ["SM_FRAMEWORK"] = "tf.keras"
from google.colab import drive
drive.mount("/content/drive")

INPUT_ROOT = Path("/content/drive/MyDrive/lostwells_unet")
REPO = Path("/content/lostwells")
if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "https://github.com/ShrishPremkrishna/lostwells.git", str(REPO)], check=True)
UNET = REPO / "UNET"
os.chdir(UNET)

MODEL = INPUT_ROOT / "lbnl" / "unet_model.h5"
WELLS = [INPUT_ROOT / "wells" / f"{s}.geojson" for s in ("PA", "WV", "OH", "KY")]
MANIFEST = UNET / "manifests" / "core_appalachia_latest.csv"
missing = [str(p) for p in [MODEL, MANIFEST, *WELLS]
           if not p.exists() or p.stat().st_size == 0]
if missing:
    raise FileNotFoundError("Missing required input(s):\n" + "\n".join(missing))
print("Master input gate: PASS")

## 3. Inspect the committed first-pass population

These counts are maps, not wells. The number of candidate wells is unknown until inference runs.

In [ ]:
from collections import Counter
with MANIFEST.open(newline="") as fh:
    manifest_rows = list(csv.DictReader(fh))
counts = Counter(r["primary_state"] for r in manifest_rows)
size_gb = sum(int(r["product_filesize"] or 0) for r in manifest_rows) / 1e9
print("Maps by state:", dict(counts))
print("Total maps:", len(manifest_rows))
print(f"Transient downloads if every map is processed: {size_gb:.1f} GB")
assert len(manifest_rows) == 2705

## 4. Optional Kern reproduction gate

Run this only when the five California reference files remain in `lbnl/`. It proves model loading, preprocessing, tiling, coordinate conversion, and thresholding against released results. Missing optional files skip this gate rather than triggering an API download.

In [ ]:
import zipfile
LBNL = INPUT_ROOT / "lbnl"
sample_tif = LBNL / "CA_OilCenter_293661_1954_24000_geo.tif"
calgem = LBNL / "CalGEM_AllWells_20231128.csv"
archive = LBNL / "found_potential_UOWs.zip"

if all(p.exists() for p in (sample_tif, calgem, archive, MODEL)):
    extract_dir = LBNL / "found_potential_UOWs"
    extract_dir.mkdir(exist_ok=True)
    with zipfile.ZipFile(archive) as zf:
        zf.extractall(extract_dir)
    published = next(extract_dir.rglob("Kern_CA_UOWs.csv"))
    subprocess.run([sys.executable, "scripts/validate_kern.py",
                    "--quads", str(sample_tif.parent), "--model", str(MODEL),
                    "--documented", str(calgem), "--published", str(published),
                    "--batch-size", "16"], check=True)
else:
    print("Optional Kern files are incomplete; skipping reproduction gate.")

## Repository and Drive layout

Code and the 2,705-map manifest live in GitHub. Large inputs and all outputs remain in Drive:

```text
MyDrive/lostwells_unet/
  lbnl/unet_model.h5
  wells/PA.geojson
  wells/WV.geojson
  wells/OH.geojson
  wells/KY.geojson
  outputs/
```

See `COLAB_RUNBOOK.md` for the complete two-computer procedure.

## Worker allocation

- Computer/account A: `run_WV_colab.ipynb`, then `run_PA_colab.ipynb`.
- Computer/account B: `run_OH_colab.ipynb`, then `run_KY_colab.ipynb`.

Run one GPU notebook per free account. Each worker is restartable.

## Final cross-state merge

Run this after all state outputs are accessible from one Drive account. Set `STATE_OUTPUT_DIRS` to the four per-map directories, not the already-merged state files.

In [ ]:
import os, subprocess, sys
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")

REPO = Path("/content/lostwells")
if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "https://github.com/ShrishPremkrishna/lostwells.git", str(REPO)], check=True)
UNET = REPO / "UNET"

# Adjust account-2 paths to the Drive shortcut you created.
STATE_OUTPUT_DIRS = [
    Path("/content/drive/MyDrive/lostwells_unet/outputs/PA"),
    Path("/content/drive/MyDrive/lostwells_unet/outputs/WV"),
    Path("/content/drive/MyDrive/account2_outputs/OH"),
    Path("/content/drive/MyDrive/account2_outputs/KY"),
]
missing = [str(p) for p in STATE_OUTPUT_DIRS if not p.exists()]
if missing:
    raise FileNotFoundError("Update STATE_OUTPUT_DIRS; missing:\n" + "\n".join(missing))

FINAL = Path("/content/drive/MyDrive/lostwells_unet/outputs/core_appalachia_candidates.geojson")
subprocess.run([sys.executable, str(UNET/"scripts"/"merge_detections.py"),
                "--inputs", *map(str, STATE_OUTPUT_DIRS),
                "--out", str(FINAL), "--radius-m", "60"], check=True)
print("Final standalone candidate file:", FINAL)